### Reweighting Technique in Sampling

This post is largely inspired by the materials in [3rd i-CoMSE Workshop](https://github.com/icomse/3rd_workshop_advanced_sampling). 

Prerequisite: Basic Molecular Dynamics, Thermodynamics, Probability Theory.

---

#### Boltzmann Distribution
Starting from the second law of thermodynamics, we can derive that, at a given temperature, the probability of finding a specific microscopic configuration $x_i$ follows the **Boltzmann distribution** (sometimes called the Gibbs distribution)

$$
    p_i \propto e^{-\beta E_i} = \frac{1}{Z} \mathrm{e}^{-\beta E_i}
$$

where $\beta = 1/(k_B T)$ is inversely proportional to the temperature of the system, and have the unit of inverse energy (it is useful to remember that at room temperature, $k_B T \approx 2.5 \,\mathrm{kJ/mol}$).

The normalization factor $Z$, called the **partition function**, is defined as

$$
    Z = \sum_{i} \mathrm{e}^{-\beta E_i}
$$

There are many interpretations of the partition function. One useful way to think about it is as a **weighted count of all microscopic configurations**. To see this, we can insert a magic one in the expression (we will use this trick many times in the post)

$$
    Z = \sum_{i} 1 \cdot \mathrm{e}^{-\beta E_i}
$$

If the exponential factor were always $1$, then $Z$ would simply count the number of accessible configurations. But the exponential factor, also called the **Boltzmann factor** or **Boltzmann weight**, assigns more importance to lower-energy states. Thus, $Z$ is not just a count, but a weighted sum that reflects how nature prefers lower-energy configurations at finite temperature.

From this formula we also see a key feature of the Boltzmann distribution: all microstates with the same energy are equally probable. In other words, the probability depends only on the energy, not on any other label of the configuration.


#### Ensemble
In statistical mechanics, an ensemble is a collection of hypothetical copies of a system that all share the same macroscopic properties (such as temperature, pressure, volume, and chemical potential) but may differ in their microscopic details (such as the positions and velocities of individual particles). The most common ensemble is the so-called canonical ensemble (also $NVT$ ensemble), where the number of particles $N$, volume $V$, and temperature $T$ are fixed. The probability distribution is given by the Boltzmann distribution.

In a molecular system, the microscopic configuration is determined by the continuous positions $\mathbf{x}$ and momenta $\mathbf{p}$ (note that if the system has $N$ particles, the dimension of both $\mathbf{x}$ and $\mathbf{p}$ is $3N$). Thus we can rewrite the Boltzmann distribution as

$$
    p(\mathbf{x}, \mathbf{p}) = 
    \frac{\mathrm{e}^{-\beta \left(U(\mathbf{x}) + K(\mathbf{p})\right)}}
         {\iint \mathrm{e}^{-\beta \left(U(\mathbf{x}) + K(\mathbf{p})\right)} \, d\mathbf{x} \, d\mathbf{p}}
$$

Most of the time, we are only interested in the configurational distribution, so we can integrate out the momenta. This gives

$$
    p(\mathbf{x}) = 
    \frac{\int \mathrm{e}^{-\beta \left(U(\mathbf{x}) + K(\mathbf{p})\right)} \, d\mathbf{p}}
         {\iint \mathrm{e}^{-\beta \left(U(\mathbf{x}) + K(\mathbf{p})\right)} \, d\mathbf{x} \, d\mathbf{p}}
    = \frac{\mathrm{e}^{-\beta U(\mathbf{x})}}
           {\int \mathrm{e}^{-\beta U(\mathbf{x})} \, d\mathbf{x}}
    = \frac{1}{Z} \, \mathrm{e}^{-\beta U(\mathbf{x})}
$$

where $Z$ is the configurational partition function. Note that $Z$ is a muitl-dimensional intergral ($3N$) which is generally impossible to evaluate.

#### Expectation

Often, we are interested in how to measure a physical observable $O$ in a given ensemble. From probability theory, we can calculate its expectation value as

$$
    \langle O \rangle = \int O(\mathbf{x}) \, p(\mathbf{x}) \, d\mathbf{x} 
    = \frac{\int O(\mathbf{x}) \, \mathrm{e}^{-\beta U(\mathbf{x})} \, d\mathbf{x}}
           {\int \mathrm{e}^{-\beta U(\mathbf{x})} \, d\mathbf{x}}
$$

However, for most systems beyond the trivial cases, evaluating this integral exactly is impossible, and we must seek approximations. Suppose that we can somehow sample from the Boltzmann distribution (we will return to sampling methods later). In that case, we can use **Monte Carlo integration** to approximate the expectation value:

$$
    \langle O \rangle = \int O(\mathbf{x}) \, p(\mathbf{x}) \, d\mathbf{x} 
    \approx \frac{1}{N} \sum_{n=1}^N O(\mathbf{x}_n),\quad\mathbf{x}_n\sim p(\mathbf{x})
$$

where each $\mathbf{x}_n$ is sampled from $p(\mathbf{x})$. It is important to note that there is no explicit Boltzmann factor in the Monte Carlo estimator, because it is already implicitly included in the sampling procedure.

Everything looks good, but the problem is that we cannot efficiently sample from an arbitrary distribution, especially for the complex, high-dimensional Boltzmann distribution. This is where the method of **importance sampling** comes in. It allows us to estimate observables under any distribution by using an easier-to-sample proposal distribution.

Suppose we want to measure an observable $O$ under distribution $p(\mathbf{x})$, and we can find another proposal distribution $q(\mathbf{x})$. Then

$$
    \langle O \rangle = \int O(\mathbf{x}) \, p(\mathbf{x}) \, d\mathbf{x} 
    = \int O(\mathbf{x}) \, p(\mathbf{x}) \, \frac{q(\mathbf{x})}{q(\mathbf{x})} \, d\mathbf{x}
    = \int O(\mathbf{x}) \, \frac{p(\mathbf{x})}{q(\mathbf{x})} \, q(\mathbf{x}) \, d\mathbf{x}
    \approx \frac{1}{N} \sum_{n=1}^N O(\mathbf{x}_n) \, \frac{p(\mathbf{x}_n)}{q(\mathbf{x}_n)}, \quad \mathbf{x}_n \sim q(\mathbf{x})
$$

This leads to an important identity:

$$
    \langle O \rangle 
    \approx \frac{1}{N} \sum_{n=1}^N O(\mathbf{x}_n), \quad \mathbf{x}_n \sim p(\mathbf{x})
    \;\;\;\; \approx \frac{1}{N} \sum_{n=1}^N O(\mathbf{x}_n) \, \frac{p(\mathbf{x}_n)}{q(\mathbf{x}_n)}, \quad \mathbf{x}_n \sim q(\mathbf{x})
$$

This means that if we want to measure an observable $O$ under distribution $p$, instead of sampling directly from $p$, we can sample from another distribution $q$. The expectation value of $O$ is then obtained as the average of $O(\mathbf{x}_n)$ weighted by the factor $\tfrac{p(\mathbf{x}_n)}{q(\mathbf{x}_n)}$. 

This is a very powerful technique, and it is valid for any observable $O$ and for any pair of distributions $p$ and $q$. The effectiveness of the method depends on how well $q$ approximates $p$. If the ratio $\tfrac{p(\mathbf{x}_n)}{q(\mathbf{x}_n)}$ fluctuates too much, the estimator will become very noisy. Of course, finding a suitable $q$ that both approximates $p$ well and is easy to sample from is not trivial — but before worrying about that, let’s first look at some examples of how to use it.




#### Removing Bias
In many method of enhanced sampling, we apply a bias potential to push the system comeover the energy barrer to explore the full configurational space. The problem is how can we remove the bias and recover the true measure of an observable. This is where we can apply the importance sampling.

Suppose we want to measure observable $O$ in distribution $p(\mathbf{x})$, which follows the Boltzmann distribution

$$
    p(\mathbf{x})=\frac{\mathrm{e}^{-\beta U(\mathbf{x})}}{Z_p}
$$

where $Z_p=\int\mathrm{e}^{-\beta U(\mathbf{x})}d\mathbf{x}$ is the partition funcion. But in simulation, we apply some bias $U_b$ to the original system, which makes us sample from another Boltzmann distribution $q(\mathbf{x})$

$$
    q(\mathbf{x})=\frac{\mathrm{e}^{-\beta (U(\mathbf{x})+U_b(\mathbf{x}))}}{Z_q}
$$

here $Z_q=\int\mathrm{e}^{-\beta (U(\mathbf{x})+U_b(\mathbf{x}))}d\mathbf{x}$.


In [5]:
import numpy as np
import matplotlib.pyplot as plt